[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrite/kernels/blob/main/labs/pallas-power.ipynb)

# Power day · where the kernel beats the compiler

**Hardware:** any Colab TPU runtime (v5e-1 or v6e-1). Run the first cell; if it installs anything, Runtime → Restart session, then Run all.

Measurement day benchmarked Pallas on XLA's home turf and reported the honest result: the compiler won. This notebook measures the other side, the workloads where the compiler structurally cannot follow: quadratic spill at long sequence, masks as loop structure, data-dependent loop bounds from ragged lengths, and dequantization fused into the matmul's inner loop. It also retunes the gate 01 kernels with dimension_semantics and wider block sweeps. Every kernel asserts correctness on-chip at a small size before any timing. The final cell prints one JSON blob to paste back.


In [ ]:
# Colab TPU images can pair jax with a libtpu that cannot read the
# Mosaic bytecode that jax emits. Install the latest matched pair; if
# pip installs or upgrades anything: Runtime -> Restart session, then Run all.
!pip install -q -U "jax[tpu]"
import importlib.metadata as md
print("jax", md.version("jax"), "· libtpu", md.version("libtpu"))


In [ ]:
import os
os.environ.pop("TPU_LIBRARY_PATH", None)
import time, json, functools
import numpy as np
import jax
import jax.numpy as jnp
from jax.experimental import pallas as pl
from jax.experimental.pallas import tpu as pltpu

print(jax.__version__, jax.devices())
ON_TPU = jax.devices()[0].platform == "tpu"
CHIP = jax.devices()[0].device_kind if ON_TPU else "none"
RESULTS = {"chip": CHIP, "notebook": "pallas-power", "results": {}}

# names that moved between jax versions
MS = getattr(pltpu, "MemorySpace", None) or getattr(pltpu, "TPUMemorySpace")
CP = getattr(pltpu, "CompilerParams", None) or getattr(pltpu, "TPUCompilerParams", None)
SMEM = MS.SMEM

def bench(fn, *args, reps=20):
    fn(*args).block_until_ready()
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter()
        fn(*args).block_until_ready()
        ts.append(time.perf_counter() - t0)
    return float(np.median(ts)) * 1e6  # us

def record(key, value):
    RESULTS["results"][key] = value
    print(f"  -> {key} = {value}")

def maxerr(a, b):
    return float(jnp.abs(a.astype(jnp.float32) - b.astype(jnp.float32)).max())

if ON_TPU:
    pl.pallas_call(
        lambda x_ref, o_ref: o_ref.__setitem__(..., x_ref[...] + 1),
        out_shape=jax.ShapeDtypeStruct((8, 128), jnp.float32),
    )(jnp.zeros((8, 128), jnp.float32)).block_until_ready()
    print("pallas smoke test: OK")


## Power 1 · the matmul, retuned


In [ ]:
# power 1: the gate 01 matmul, retuned. dimension_semantics tells Mosaic
# which grid axes are safe to parallelize; the K axis stays arbitrary
# because it carries the accumulator.
def matmul_kernel(a_ref, b_ref, o_ref):
    k = pl.program_id(2)
    @pl.when(k == 0)
    def _():
        o_ref[...] = jnp.zeros_like(o_ref)
    o_ref[...] += jnp.dot(a_ref[...], b_ref[...], preferred_element_type=jnp.float32).astype(o_ref.dtype)

def matmul(a, b, bm, bn, bk):
    m, k = a.shape
    _, n = b.shape
    kwargs = {}
    if CP is not None:
        kwargs["compiler_params"] = CP(dimension_semantics=("parallel", "parallel", "arbitrary"))
    return pl.pallas_call(
        matmul_kernel,
        grid=(m // bm, n // bn, k // bk),
        in_specs=[pl.BlockSpec((bm, bk), lambda i, j, kk: (i, kk)),
                  pl.BlockSpec((bk, bn), lambda i, j, kk: (kk, j))],
        out_specs=pl.BlockSpec((bm, bn), lambda i, j, kk: (i, j)),
        out_shape=jax.ShapeDtypeStruct((m, n), a.dtype),
        **kwargs,
    )(a, b)

if ON_TPU:
    N = 4096
    a = jax.random.normal(jax.random.key(0), (N, N), jnp.bfloat16)
    b = jax.random.normal(jax.random.key(1), (N, N), jnp.bfloat16)
    assert maxerr(matmul(a[:512, :512], b[:512, :512], 256, 256, 256),
                  a[:512, :512] @ b[:512, :512]) < 1.0
    xla_us = bench(jax.jit(lambda x, y: x @ y), a, b)
    best = None
    for bm, bn, bk in [(512, 1024, 512), (1024, 512, 512), (512, 512, 1024),
                       (1024, 1024, 512), (512, 1024, 1024), (1024, 1024, 1024),
                       (2048, 1024, 512), (1024, 2048, 512)]:
        try:
            us = bench(jax.jit(functools.partial(matmul, bm=bm, bn=bn, bk=bk)), a, b)
            print(f"  ({bm},{bn},{bk}): {us:8.0f} us  ratio {us / xla_us:5.2f}x")
            if best is None or us < best[1]:
                best = ((bm, bn, bk), us)
        except Exception as e:
            print(f"  ({bm},{bn},{bk}): failed: {str(e)[:80]}")
    record("power/matmul_tuned", {"xla_us": round(xla_us, 1), "pallas_best_us": round(best[1], 1),
                                  "best_block": list(best[0]), "ratio": round(best[1] / xla_us, 3),
                                  "dimension_semantics": CP is not None})


## Power 2 · the softmax, retuned


In [ ]:
# power 2: the gate 01 softmax, retuned: bigger row blocks amortize the
# per-step cost that sank the rows=64 schedule.
def softmax_kernel(x_ref, o_ref):
    x = x_ref[...].astype(jnp.float32)
    m = jnp.max(x, axis=-1, keepdims=True)
    e = jnp.exp(x - m)
    o_ref[...] = (e / jnp.sum(e, axis=-1, keepdims=True)).astype(o_ref.dtype)

def softmax(x, rows):
    n, d = x.shape
    kwargs = {}
    if CP is not None:
        kwargs["compiler_params"] = CP(dimension_semantics=("parallel",))
    return pl.pallas_call(
        softmax_kernel,
        grid=(n // rows,),
        in_specs=[pl.BlockSpec((rows, d), lambda i: (i, 0))],
        out_specs=pl.BlockSpec((rows, d), lambda i: (i, 0)),
        out_shape=jax.ShapeDtypeStruct(x.shape, x.dtype),
        **kwargs,
    )(x)

if ON_TPU:
    big = jax.random.normal(jax.random.key(0), (32768, 512), jnp.bfloat16)
    assert maxerr(softmax(big[:512], 128), jax.nn.softmax(big[:512].astype(jnp.float32), -1)) < 2e-2
    m_fn = jax.jit(lambda v: jnp.max(v, -1, keepdims=True))
    e_fn = jax.jit(jnp.exp)
    s_fn = jax.jit(lambda v: jnp.sum(v, -1, keepdims=True))
    def unfused(x):
        m = m_fn(x); e = e_fn(x - m); return e / s_fn(e)
    unf = bench(unfused, big)
    xla = bench(jax.jit(lambda v: jax.nn.softmax(v, -1)), big)
    best = None
    for rows in [128, 256, 512, 1024, 2048]:
        us = bench(jax.jit(functools.partial(softmax, rows=rows)), big)
        print(f"  rows={rows}: {us:8.0f} us")
        if best is None or us < best[1]:
            best = (rows, us)
    record("power/softmax_tuned", {"unfused_us": round(unf, 1), "xla_fused_us": round(xla, 1),
                                   "pallas_best_us": round(best[1], 1), "best_rows": best[0],
                                   "beats_unfused": bool(best[1] < unf)})


## Power 3 · the scaling curve: flash vs naive, seq 4k to 32k


In [ ]:
# power 3: the scaling curve. Naive's HBM traffic grows with seq^2; flash
# stays linear in memory. One number per sequence length, same kernels.
def naive_attention(q, k, v):
    s = q @ k.T
    m = jnp.max(s, axis=-1, keepdims=True)
    p = jnp.exp(s - m)
    return (p / jnp.sum(p, axis=-1, keepdims=True)) @ v

def flash_kernel(q_ref, k_ref, v_ref, o_ref, *, block_kv):
    q = q_ref[...].astype(jnp.float32)
    n_kv = k_ref.shape[0]
    def step(j, state):
        m, l, acc = state
        kb = k_ref[pl.ds(j * block_kv, block_kv), :].astype(jnp.float32)
        vb = v_ref[pl.ds(j * block_kv, block_kv), :].astype(jnp.float32)
        s = q @ kb.T
        m_new = jnp.maximum(m, jnp.max(s, axis=-1))
        alpha = jnp.exp(m - m_new)
        p = jnp.exp(s - m_new[:, None])
        return m_new, l * alpha + jnp.sum(p, axis=-1), acc * alpha[:, None] + p @ vb
    m0 = jnp.full((q.shape[0],), -jnp.inf, jnp.float32)
    l0 = jnp.zeros((q.shape[0],), jnp.float32)
    acc0 = jnp.zeros((q.shape[0], v_ref.shape[1]), jnp.float32)
    m, l, acc = jax.lax.fori_loop(0, n_kv // block_kv, step, (m0, l0, acc0))
    o_ref[...] = (acc / l[:, None]).astype(o_ref.dtype)

def flash(q, k, v, block_q=512, block_kv=1024):
    sq, d = q.shape
    return pl.pallas_call(
        functools.partial(flash_kernel, block_kv=block_kv),
        grid=(sq // block_q,),
        in_specs=[pl.BlockSpec((block_q, d), lambda i: (i, 0)),
                  pl.BlockSpec(k.shape, lambda i: (0, 0)),
                  pl.BlockSpec(v.shape, lambda i: (0, 0))],
        out_specs=pl.BlockSpec((block_q, d), lambda i: (i, 0)),
        out_shape=jax.ShapeDtypeStruct(q.shape, q.dtype),
    )(q, k, v)

if ON_TPU:
    D = 128
    qs = jax.random.normal(jax.random.key(0), (1024, D), jnp.bfloat16)
    assert maxerr(jax.jit(flash)(qs, qs, qs), jax.jit(naive_attention)(qs, qs, qs)) < 0.5
    rows = []
    for S in [4096, 8192, 16384, 32768]:
        q = jax.random.normal(jax.random.key(0), (S, D), jnp.bfloat16)
        k = jax.random.normal(jax.random.key(1), (S, D), jnp.bfloat16)
        v = jax.random.normal(jax.random.key(2), (S, D), jnp.bfloat16)
        flash_us = bench(jax.jit(flash), q, k, v)
        try:
            naive_us = bench(jax.jit(naive_attention), q, k, v)
        except Exception as e:
            naive_us = None
            print(f"  seq {S}: naive failed: {str(e)[:60]}")
        try:
            ref_fast = jax.jit(lambda q, k, v: jax.nn.dot_product_attention(
                q[None, :, None], k[None, :, None], v[None, :, None])[0, :, 0])
            ref_us = bench(ref_fast, q, k, v)
        except Exception:
            ref_us = None
        row = {"seq": S, "flash_us": round(flash_us, 1),
               "naive_us": round(naive_us, 1) if naive_us else None,
               "reference_us": round(ref_us, 1) if ref_us else None,
               "flash_vs_naive": round(naive_us / flash_us, 2) if naive_us else None}
        rows.append(row)
        print(f"  {row}")
    record("power/flash_scaling", rows)


## Power 4 · the mask as loop structure: windowed attention at 16k


In [ ]:
# power 4: the mask as loop structure, at scale. A local attention window
# visits ~window/seq of the blocks; XLA's masked version still pays for
# the full score matrix.
def windowed_kernel(q_ref, k_ref, v_ref, o_ref, *, block_q, block_kv, window):
    qi = pl.program_id(0)
    q = q_ref[...].astype(jnp.float32)
    row0 = qi * block_q
    def step(j, state):
        m, l, acc = state
        kb = k_ref[pl.ds(j * block_kv, block_kv), :].astype(jnp.float32)
        vb = v_ref[pl.ds(j * block_kv, block_kv), :].astype(jnp.float32)
        s = q @ kb.T
        cols = j * block_kv + jax.lax.broadcasted_iota(jnp.int32, s.shape, 1)
        rows = row0 + jax.lax.broadcasted_iota(jnp.int32, s.shape, 0)
        live = (cols <= rows) & (cols > rows - window)
        s = jnp.where(live, s, -jnp.inf)
        m_new = jnp.maximum(m, jnp.max(s, axis=-1))
        alpha = jnp.where(jnp.isneginf(m), 0.0, jnp.exp(m - m_new))
        p = jnp.where(jnp.isneginf(s), 0.0, jnp.exp(s - m_new[:, None]))
        return m_new, l * alpha + jnp.sum(p, axis=-1), acc * alpha[:, None] + p @ vb
    lo = jnp.maximum(row0 - window + 1, 0) // block_kv
    hi = (row0 + block_q + block_kv - 1) // block_kv
    m0 = jnp.full((block_q,), -jnp.inf, jnp.float32)
    l0 = jnp.zeros((block_q,), jnp.float32)
    acc0 = jnp.zeros((block_q, v_ref.shape[1]), jnp.float32)
    m, l, acc = jax.lax.fori_loop(lo, hi, step, (m0, l0, acc0))
    o_ref[...] = (acc / jnp.maximum(l, 1e-30)[:, None]).astype(o_ref.dtype)

def windowed(q, k, v, window, block_q=512, block_kv=512):
    sq, d = q.shape
    return pl.pallas_call(
        functools.partial(windowed_kernel, block_q=block_q, block_kv=block_kv, window=window),
        grid=(sq // block_q,),
        in_specs=[pl.BlockSpec((block_q, d), lambda i: (i, 0)),
                  pl.BlockSpec(k.shape, lambda i: (0, 0)),
                  pl.BlockSpec(v.shape, lambda i: (0, 0))],
        out_specs=pl.BlockSpec((block_q, d), lambda i: (i, 0)),
        out_shape=jax.ShapeDtypeStruct(q.shape, q.dtype),
    )(q, k, v)

def xla_windowed(q, k, v, window):
    S = q.shape[0]
    s = (q @ k.T).astype(jnp.float32)
    rows = jnp.arange(S)[:, None]
    cols = jnp.arange(S)[None, :]
    s = jnp.where((cols <= rows) & (cols > rows - window), s, -jnp.inf)
    return (jax.nn.softmax(s, axis=-1) @ v.astype(jnp.float32)).astype(q.dtype)

if ON_TPU:
    S, D, W = 16384, 128, 1024
    small = jax.random.normal(jax.random.key(0), (1024, D), jnp.bfloat16)
    assert maxerr(jax.jit(functools.partial(windowed, window=256))(small, small, small),
                  jax.jit(functools.partial(xla_windowed, window=256))(small, small, small)) < 0.5
    q = jax.random.normal(jax.random.key(0), (S, D), jnp.bfloat16)
    k = jax.random.normal(jax.random.key(1), (S, D), jnp.bfloat16)
    v = jax.random.normal(jax.random.key(2), (S, D), jnp.bfloat16)
    win_us = bench(jax.jit(functools.partial(windowed, window=W)), q, k, v)
    try:
        xla_us = bench(jax.jit(functools.partial(xla_windowed, window=W)), q, k, v)
    except Exception as e:
        xla_us = None
        print(f"  xla masked failed: {str(e)[:60]}")
    dense_us = bench(jax.jit(flash), q, k, v)
    record("power/windowed", {"seq": S, "window": W, "pallas_us": round(win_us, 1),
                              "xla_masked_us": round(xla_us, 1) if xla_us else None,
                              "dense_flash_us": round(dense_us, 1),
                              "vs_xla_masked": round(xla_us / win_us, 2) if xla_us else None,
                              "vs_dense_flash": round(dense_us / win_us, 2)})


## Power 5 · ragged lengths: the loop bound comes from data


In [ ]:
# power 5: data-dependent loop bounds. Each sequence's true length lives
# in SMEM and bounds the kv loop; XLA must pad every sequence to the max.
# This is the splash / ragged-paged-attention mechanism, taught small.
def ragged_kernel(len_ref, q_ref, k_ref, v_ref, o_ref, *, block_q, block_kv):
    b = pl.program_id(0)
    qi = pl.program_id(1)
    length = len_ref[b]
    q = q_ref[0].astype(jnp.float32)
    row0 = qi * block_q
    def step(j, state):
        m, l, acc = state
        kb = k_ref[0, pl.ds(j * block_kv, block_kv), :].astype(jnp.float32)
        vb = v_ref[0, pl.ds(j * block_kv, block_kv), :].astype(jnp.float32)
        s = q @ kb.T
        cols = j * block_kv + jax.lax.broadcasted_iota(jnp.int32, s.shape, 1)
        s = jnp.where(cols < length, s, -jnp.inf)
        m_new = jnp.maximum(m, jnp.max(s, axis=-1))
        alpha = jnp.where(jnp.isneginf(m), 0.0, jnp.exp(m - m_new))
        p = jnp.where(jnp.isneginf(s), 0.0, jnp.exp(s - m_new[:, None]))
        return m_new, l * alpha + jnp.sum(p, axis=-1), acc * alpha[:, None] + p @ vb
    n_live = (length + block_kv - 1) // block_kv
    m0 = jnp.full((block_q,), -jnp.inf, jnp.float32)
    l0 = jnp.zeros((block_q,), jnp.float32)
    acc0 = jnp.zeros((block_q, v_ref.shape[2]), jnp.float32)
    m, l, acc = jax.lax.fori_loop(0, n_live, step, (m0, l0, acc0))
    out = acc / jnp.maximum(l, 1e-30)[:, None]
    rows = row0 + jax.lax.broadcasted_iota(jnp.int32, out.shape, 0)
    out = jnp.where(rows < length, out, 0.0)
    o_ref[0] = out.astype(o_ref.dtype)

def ragged(lengths, q, k, v, block_q=512, block_kv=512):
    B, S, d = q.shape
    return pl.pallas_call(
        functools.partial(ragged_kernel, block_q=block_q, block_kv=block_kv),
        grid=(B, S // block_q),
        in_specs=[pl.BlockSpec(memory_space=SMEM),
                  pl.BlockSpec((1, block_q, d), lambda b, i: (b, i, 0)),
                  pl.BlockSpec((1, S, d), lambda b, i: (b, 0, 0)),
                  pl.BlockSpec((1, S, d), lambda b, i: (b, 0, 0))],
        out_specs=pl.BlockSpec((1, block_q, d), lambda b, i: (b, i, 0)),
        out_shape=jax.ShapeDtypeStruct(q.shape, q.dtype),
    )(lengths, q, k, v)

def xla_padded(lengths, q, k, v):
    B, S, d = q.shape
    s = jnp.einsum("bqd,bkd->bqk", q, k).astype(jnp.float32)
    cols = jnp.arange(S)[None, None, :]
    s = jnp.where(cols < lengths[:, None, None], s, -jnp.inf)
    o = jnp.einsum("bqk,bkd->bqd", jax.nn.softmax(s, axis=-1), v.astype(jnp.float32))
    rows = jnp.arange(S)[None, :, None]
    return jnp.where(rows < lengths[:, None, None], o, 0.0).astype(q.dtype)

if ON_TPU:
    B, S, D = 8, 4096, 128
    q = jax.random.normal(jax.random.key(0), (B, S, D), jnp.bfloat16)
    k = jax.random.normal(jax.random.key(1), (B, S, D), jnp.bfloat16)
    v = jax.random.normal(jax.random.key(2), (B, S, D), jnp.bfloat16)
    lengths = jnp.array([512, 3584, 1024, 256, 2048, 512, 4096, 1536], jnp.int32)
    assert maxerr(jax.jit(ragged)(lengths, q, k, v), jax.jit(xla_padded)(lengths, q, k, v)) < 0.5
    rag_us = bench(jax.jit(ragged), lengths, q, k, v)
    pad_us = bench(jax.jit(xla_padded), lengths, q, k, v)
    frac = float(jnp.mean(lengths / S))
    record("power/ragged", {"batch": B, "seq_max": S, "mean_len_frac": round(frac, 3),
                            "padded_xla_us": round(pad_us, 1), "ragged_pallas_us": round(rag_us, 1),
                            "speedup": round(pad_us / rag_us, 2)})


## Power 6 · dequantization fused into the matmul


In [ ]:
# power 6: dequantization fused into the kernel. Decode-shaped matmuls are
# bandwidth-bound, and int8 weights halve the bytes only if the dequant
# never round-trips a bf16 copy through HBM.
def dequant_kernel(x_ref, wq_ref, s_ref, o_ref):
    w = wq_ref[...].astype(jnp.float32) * s_ref[...].astype(jnp.float32)[None, :]
    o_ref[...] = jnp.dot(x_ref[...].astype(jnp.float32), w,
                         preferred_element_type=jnp.float32).astype(o_ref.dtype)

def dequant_matmul(x, wq, scale, bn):
    m, kk = x.shape
    _, n = wq.shape
    return pl.pallas_call(
        dequant_kernel,
        grid=(n // bn,),
        in_specs=[pl.BlockSpec((m, kk), lambda j: (0, 0)),
                  pl.BlockSpec((kk, bn), lambda j: (0, j)),
                  pl.BlockSpec((bn,), lambda j: (j,))],
        out_specs=pl.BlockSpec((m, bn), lambda j: (0, j)),
        out_shape=jax.ShapeDtypeStruct((m, n), x.dtype),
    )(x, wq, scale)

if ON_TPU:
    M, K, N = 8, 4096, 14336
    x = jax.random.normal(jax.random.key(0), (M, K), jnp.bfloat16)
    wq = jax.random.randint(jax.random.key(1), (K, N), -127, 127, jnp.int8)
    scale = (jnp.abs(jax.random.normal(jax.random.key(2), (N,))) * 0.01).astype(jnp.bfloat16)
    xla_fn = jax.jit(lambda x, wq, s: x @ (wq.astype(jnp.bfloat16) * s))
    assert maxerr(dequant_matmul(x[:, :256], wq[:256, :512], scale[:512], 512),
                  x[:, :256] @ (wq[:256, :512].astype(jnp.bfloat16) * scale[:512])) < 1.0
    xla_us = bench(xla_fn, x, wq, scale)
    bf16_us = bench(jax.jit(lambda x, w: x @ w), x, (wq.astype(jnp.bfloat16) * scale))
    best = None
    for bn in [512, 1024, 2048]:
        us = bench(jax.jit(functools.partial(dequant_matmul, bn=bn)), x, wq, scale)
        print(f"  bn={bn}: {us:8.0f} us")
        if best is None or us < best[1]:
            best = (bn, us)
    record("power/int8_decode", {"xla_dequant_us": round(xla_us, 1),
                                 "bf16_weights_us": round(bf16_us, 1),
                                 "pallas_fused_us": round(best[1], 1), "best_bn": best[0],
                                 "vs_xla_dequant": round(xla_us / best[1], 2),
                                 "vs_bf16": round(bf16_us / best[1], 2)})


## The blob


In [ ]:
print("=" * 60)
print("POWER DAY RESULTS · paste this whole blob back")
print("=" * 60)
print(json.dumps(RESULTS, indent=1))
